<a href="https://colab.research.google.com/github/Johnogunlola/MRes-AI/blob/MRes/AI_DRIVEN_FLIGHT_DELAY_PREDICTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Install required libraries
!pip install pandas numpy matplotlib seaborn scikit-learn lightgbm xgboost catboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

print("Libraries loaded.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.3 MB/s eta 0:00:00
Libraries loaded.


In [ ]:
# Read the dataset
filename = next(iter(uploaded))
df = pd.read_csv(2019-2024_Annual_Flight_Statistics_for_UK_Airports.csv)

print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns")
display(df.head())

SyntaxError: invalid decimal literal (ipython-input-2-691940359.py, line 3)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Read the dataset
filename = next(iter(uploaded))
df = pd.read_csv(filename)

print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns")
display(df.head())

In [ ]:
# Drop irrelevant or empty columns
df.dropna(how='all', axis=1, inplace=True)
df.drop(['Unnamed: 17'], axis=1, errors='ignore', inplace=True)  # Example of known empty column

# Fill missing values
df.fillna(0, inplace=True)

# Rename columns for clarity
df.rename(columns={
    'Reporting Year': 'Year',
    'Reporting Airport': 'Airport',
    'Flights 0 (zero) to 15 minutes late percent': 'OnTime',
    'Flights between 16 and 30 minutes late percent': 'Delay_16_30',
    'Flights between 31 and 60 minutes late percent': 'Delay_31_60',
    'Flights between 61 and 120 minutes late percent': 'Delay_61_120',
    'Flights between 121 and 180 minutes late percent': 'Delay_121_180',
    'Flights between 181 and 360 minutes late percent': 'Delay_181_360',
    'Flights more than 360 minutes late percent': 'Delay_Over_360',
    'Flights Cancelled Percent': 'Canceled' # Corrected column name
}, inplace=True)

# Combine all delays >15 minutes
delay_columns = [
    'Delay_16_30', 'Delay_31_60', 'Delay_61_120',
    'Delay_121_180', 'Delay_181_360', 'Delay_Over_360'
]
df['Total_Delay'] = df[delay_columns].sum(axis=1)
df['is_delayed'] = df['Total_Delay'].apply(lambda x: 1 if x > 0 else 0)
df['is_canceled'] = df['Canceled'].apply(lambda x: 1 if x > 0 else 0)

# Encode airport names
le = LabelEncoder()
df['Airport_Code'] = le.fit_transform(df['Airport'])

# Select features and targets
features = ['Year', 'Airport_Code', 'Number of Flights', 'OnTime', 'Total_Delay', 'Canceled'] # Corrected column name
X = df[['Year', 'Airport_Code', 'Number of Flights']]
y_delay = df['is_delayed']
y_cancel = df['is_canceled']

print("Data cleaned and features engineered.")

In [ ]:
# Distribution of delays vs on-time flights
plt.figure(figsize=(8, 4))
sns.countplot(x='is_delayed', data=df)
plt.title('Class Distribution: Delayed (1) vs On-Time (0)')
plt.show()

# Correlation matrix
corr = df[['Number of Flights', 'OnTime', 'Total_Delay', 'Canceled']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_delay, test_size=0.2, random_state=42, stratify=y_delay
)

In [ ]:
# Define models
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "LightGBM": LGBMClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False),
    "CatBoost": CatBoostClassifier(silent=True)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    print(f"\n{name} - Accuracy: {acc:.4f}, AUC: {auc:.4f}")
    print(classification_report(y_test, y_pred))
    results[name] = {'Accuracy': acc, 'AUC': auc}

In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame(results).T
results_df.sort_values(by='Accuracy', ascending=False, inplace=True)

print("\nModel Performance Summary:")
print(results_df)

# Plot performance comparison
results_df.plot(kind='bar', figsize=(10, 6), title="Model Accuracy & AUC Comparison")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_cancel, test_size=0.2, random_state=42, stratify=y_cancel
)

model_cancel = RandomForestClassifier()
model_cancel.fit(X_train_c, y_train_c)
y_pred_c = model_cancel.predict(X_test_c)

print("Cancellation Prediction Report:")
print(classification_report(y_test_c, y_pred_c))

In [ ]:
# Step 1: Install SHAP if not already installed
!pip install shap

import shap

print("SHAP library loaded.")

In [ ]:
from lightgbm import LGBMClassifier

# Use full dataset again for better interpretation
X_full = df[['Year', 'Airport_Code', 'Number of Flights']]
y_full = df['is_delayed']

# Train final model
model_lgb = LGBMClassifier()
model_lgb.fit(X_full, y_full)

print("Model trained for SHAP explanation.")

In [ ]:
# Create a SHAP explainer
explainer = shap.Explainer(model_lgb)
shap_values = explainer.shap_values(X_full)

print("SHAP values computed.")

In [ ]:
# Summary plot: overall feature importance
shap.summary_plot(shap_values, X_full, feature_names=X_full.columns)

In [ ]:
# Bar plot: average absolute SHAP value per feature
shap.summary_plot(shap_values, X_full, feature_names=X_full.columns, plot_type="bar")

In [ ]:
# Select a sample instance (e.g., first row)
sample_idx = 0
shap.force_plot(explainer.expected_value[1], shap_values[1][sample_idx], X_full.iloc[sample_idx], feature_names=X_full.columns)

In [ ]:
# Identify the most important feature from the summary plot (using the correct method)
most_important_feature = X_full.columns[np.argmax(np.abs(shap_values[1]).mean(0))]

# Generate a dependence plot for the most important feature
shap.dependence_plot(most_important_feature, shap_values[1], X_full, interaction_index=None)

In [ ]:
shap.dependence_plot("Number of Flights", shap_values[1], X_full)

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn lightgbm xgboost catboost lime shap

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Upload your dataset
from google.colab import files
uploaded = files.upload()

# Read the dataset
filename = next(iter(uploaded))
df = pd.read_csv(filename)

# Rename columns for clarity
df.rename(columns={
    'Reporting Year': 'Year',
    'Reporting Airport': 'Airport',
    'Flights 0 (zero) to 15 minutes late percent': 'OnTime',
    'Flights between 16 and 30 minutes late percent': 'Delay_16_30',
    'Flights between 31 and 60 minutes late percent': 'Delay_31_60',
    'Flights between 61 and 120 minutes late percent': 'Delay_61_120',
    'Flights between 121 and 180 minutes late percent': 'Delay_121_180',
    'Flights between 181 and 360 minutes late percent': 'Delay_181_360',
    'Flights more than 360 minutes late percent': 'Delay_Over_360',
    'Cancelled flights percent': 'Canceled'
}, inplace=True)

# Combine all delays >15 minutes
delay_columns = [
    'Delay_16_30', 'Delay_31_60', 'Delay_61_120',
    'Delay_121_180', 'Delay_181_360', 'Delay_Over_360'
]
df['Total_Delay'] = df[delay_columns].sum(axis=1)
df['is_delayed'] = df['Total_Delay'].apply(lambda x: 1 if x > 0 else 0)

# Encode airport names
le = LabelEncoder()
df['Airport_Code'] = le.fit_transform(df['Airport'])

# Select features and target
X = df[['Year', 'Airport_Code', 'Number of Flights']]
y = df['is_delayed']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Data loaded and prepared.")

In [ ]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

# Define models
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "LightGBM": LGBMClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False),
    "CatBoost": CatBoostClassifier(silent=True)
}

# Train and predict
model_pipelines = {}
roc_data = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_pred)
    roc_data[name] = (fpr, tpr, auc)
    model_pipelines[name] = model

In [ ]:
import matplotlib.pyplot as plt

# Plot ROC curve
plt.figure(figsize=(10, 7))
for name, (fpr, tpr, auc) in roc_data.items():
    plt.plot(fpr, tpr, lw=2, label='{} (AUC = {:.4f})'.format(name, auc))

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

In [ ]:
import lime
import lime.lime_tabular

# Pick one model for explanation (e.g., LightGBM)
explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train.values,
    feature_names=X.columns.tolist(),
    class_names=['On-Time', 'Delayed'],
    mode='classification'
)

# Explain a sample prediction
model_name = "LightGBM"
model = model_pipelines[model_name]
idx = 0  # Change this index to explain different instances
exp = explainer.explain_instance(X_test.iloc[idx], model.predict_proba, num_features=3)

# Show plot
print(f"\nExplanation for instance {idx} using {model_name}:")
exp.show_in_notebook(show_table=True)